# Information Theory

## What's covered

- **Entropy** — the average "surprise" of a random variable; the optimal compression bound
- **Cross-entropy** — average surprise of using model `q` to describe data from `p`
- **KL divergence** — distance from `q` to `p` in information-theoretic units
- **Mutual information** — how much knowing `X` tells you about `Y`
- **Jensen's inequality** — the convexity argument behind half of probability proofs
- The bridge from information theory to ML loss functions
- Where this appears in ML — every classification loss, every variational method, every contrastive objective


## Entropy — the average surprise

The **entropy** of a discrete random variable `X` with PMF `p` is

$$
H(X) = -\sum_x p(x) \log p(x) = \mathbb{E}[-\log p(X)]
$$

(For continuous `X`, the **differential entropy** uses an integral instead of a sum. Some properties differ, but the ML applications focus on the discrete case for classification.)

The base of the logarithm chooses the units:

- `\log_2` → bits
- `\ln` (natural log) → nats (used in ML almost universally)
- `\log_{10}` → dits or hartleys (rare)

We'll use natural log throughout, matching most ML code.

**Three ways to read entropy.**

- **Surprise.** `-\log p(x)` is the *surprise* of seeing outcome `x`. Rare events (small `p(x)`) are surprising; certain events (`p(x) = 1`) carry zero surprise. Entropy is the *expected* surprise.
- **Uncertainty.** Entropy is maximum when all outcomes are equally likely (uniform distribution) and zero when one outcome is certain. High entropy = high uncertainty; low entropy = predictable.
- **Compression bound.** Shannon's coding theorem: the optimal lossless code for outcomes from distribution `p` uses on average `H(X)` bits per symbol. You cannot compress below entropy; you should aim to get close to it.

**Examples.**

- Fair coin: `H = -0.5 \log 0.5 - 0.5 \log 0.5 = \log 2 = 1 \text{ bit}`. Maximum uncertainty for a binary variable.
- Biased coin with `p = 0.9`: `H = -0.9 \log 0.9 - 0.1 \log 0.1 \approx 0.47 \text{ bits}`. Much less uncertainty; we mostly expect heads.
- Deterministic `p = 1.0`: `H = 0`. No uncertainty.

**Properties.**

- `H(X) \ge 0` — entropy is non-negative.
- `H(X) \le \log K` for `K` outcomes; equality iff uniform. **Uniform distributions are maximum entropy.**
- `H(X, Y) \le H(X) + H(Y)`, with equality iff `X` and `Y` are independent (subadditivity).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(0)

def entropy(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]            # skip zeros, since 0 log 0 = 0 by convention
    return -np.sum(p * np.log(p))   # nats

# Bernoulli entropy as a function of p
ps = np.linspace(0.001, 0.999, 200)
H_bern = [entropy([p, 1 - p]) for p in ps]

plt.figure(figsize=(7, 3))
plt.plot(ps, H_bern)
plt.axvline(0.5, color='r', linestyle='--', label="max at p = 0.5")
plt.xlabel("p"); plt.ylabel("H(X) in nats")
plt.title("Bernoulli entropy"); plt.legend()
plt.show()

print(f"H(fair coin)   = {entropy([0.5, 0.5]):.4f} nats = {entropy([0.5, 0.5])/np.log(2):.4f} bits")
print(f"H(p = 0.9)     = {entropy([0.9, 0.1]):.4f} nats")
print(f"H(p = 1.0)     = {entropy([1.0, 0.0]):.4f} nats (no uncertainty)")
print(f"H(uniform K=6) = {entropy([1/6]*6):.4f} nats = log(6) = {np.log(6):.4f}")


## Cross-entropy

Suppose data comes from `p`, but you describe it (e.g., compress, predict) with a different distribution `q`. The **cross-entropy** is the expected surprise of `q` evaluated on `p`'s samples:

$$
H(p, q) = -\sum_x p(x) \log q(x) = \mathbb{E}_{X \sim p}[-\log q(X)]
$$

When `q = p`, cross-entropy equals entropy: `H(p, p) = H(p)`. Otherwise, `H(p, q) > H(p)` — using the wrong distribution to describe data costs extra surprise.

**The classification connection.** In supervised learning, `p` is the *empirical* distribution of labels (often a one-hot vector for a single example, putting all probability on the true class `y`) and `q` is the *model's* prediction `\hat{\mathbf{p}}`. The cross-entropy loss for one example is:

$$
\text{CE}(\mathbf{y}, \hat{\mathbf{p}}) = -\sum_k y_k \log \hat{p}_k
$$

For a one-hot target with `y_t = 1` (true class `t`), this collapses to `-\log \hat{p}_t`. **The single number** — negative log of the predicted probability of the correct class. This is the loss every classification network minimizes.

It's also exactly the **negative log-likelihood** under a Categorical model. Cross-entropy and NLL are the same thing. Different names, same object.

**Binary case.** For Bernoulli with predicted probability `\hat{p}` and label `y \in \{0, 1\}`:

$$
\text{BCE}(y, \hat{p}) = -y \log \hat{p} - (1 - y) \log(1 - \hat{p})
$$

This is what `nn.BCELoss` computes.


## KL divergence — "distance" from `q` to `p`

The most-used quantity in probabilistic ML. Defined as

$$
D_{\text{KL}}(p \,\|\, q) = \sum_x p(x) \log \frac{p(x)}{q(x)} = \mathbb{E}_{X \sim p}\!\left[\log \frac{p(X)}{q(X)}\right]
$$

Equivalent rewrite (this is the form to memorize):

$$
\boxed{D_{\text{KL}}(p \,\|\, q) = H(p, q) - H(p)}
$$

**KL divergence is the *extra* surprise from using `q` instead of `p`.** Cross-entropy minus entropy. Always non-negative. Zero iff `p = q`.

**Important properties — and one famous non-property.**

- `D_{\text{KL}}(p \,\|\, q) \ge 0` (**Gibbs' inequality**). Proved using Jensen's inequality on the concave log function.
- `D_{\text{KL}}(p \,\|\, q) = 0` iff `p = q` almost everywhere.
- **Not symmetric.** `D_{\text{KL}}(p \,\|\, q) \neq D_{\text{KL}}(q \,\|\, p)` in general. This is why KL is called a "divergence," not a "distance" — it doesn't satisfy the distance axioms. The asymmetry matters: `D_{\text{KL}}(p \,\|\, q)` penalizes `q` for having low mass where `p` has high mass (mode-covering); `D_{\text{KL}}(q \,\|\, p)` does the opposite (mode-seeking).
- **Does not satisfy the triangle inequality.**

**The classification loss connection.** For one-hot labels (where `H(p) = 0`):

$$
\text{Cross-entropy} = D_{\text{KL}}(p \,\|\, q) + H(p) = D_{\text{KL}}(p \,\|\, q)
$$

So minimizing cross-entropy = minimizing KL = pulling the predicted distribution toward the true distribution. Three different framings of the same objective.

**The variational connection.** Variational inference, VAEs, KL-regularized RL, RLHF, distillation — all use KL as the unit of distance between probability distributions.

**A subtle point.** `D_{\text{KL}}(p \,\|\, q)` is undefined (or infinite) wherever `q(x) = 0` but `p(x) > 0`. This is why classification losses often clamp predicted probabilities to `[\epsilon, 1 - \epsilon]` — to avoid `-\log 0 = \infty`.


In [ ]:
def kl_divergence(p, q):
    p = np.asarray(p, dtype=float); q = np.asarray(q, dtype=float)
    mask = p > 0
    return np.sum(p[mask] * np.log(p[mask] / q[mask]))

def cross_entropy(p, q):
    p = np.asarray(p, dtype=float); q = np.asarray(q, dtype=float)
    mask = p > 0
    return -np.sum(p[mask] * np.log(q[mask]))

# Two distributions over 4 outcomes
p = np.array([0.5, 0.25, 0.15, 0.10])
q = np.array([0.3, 0.3,  0.2,  0.20])

H_p   = entropy(p)
H_pq  = cross_entropy(p, q)
KL_pq = kl_divergence(p, q)
KL_qp = kl_divergence(q, p)

print(f"H(p)            = {H_p:.4f}")
print(f"H(p, q)         = {H_pq:.4f}")
print(f"D_KL(p || q)    = {KL_pq:.4f}")
print(f"D_KL(q || p)    = {KL_qp:.4f}   <- not equal — KL is asymmetric")
print(f"\nCheck: H(p, q) - H(p) = {H_pq - H_p:.4f}  vs  D_KL(p || q) = {KL_pq:.4f}")


## Mutual information

How much does knowing `Y` tell you about `X`? The answer is the **mutual information**:

$$
I(X; Y) = D_{\text{KL}}(p(x, y) \,\|\, p(x) p(y))
$$

It's the KL divergence between the **joint** and the **product of marginals**. If `X` and `Y` are independent, the joint *equals* the product of marginals and `I(X; Y) = 0`. Otherwise it's positive — measuring how far from independent the two variables are.

**Equivalent expressions** (great to memorize):

$$
I(X; Y) = H(X) - H(X | Y) = H(Y) - H(Y | X) = H(X) + H(Y) - H(X, Y)
$$

Reading them: mutual information is the *reduction in uncertainty* about `X` once you observe `Y` (or vice versa). It's also symmetric — `I(X; Y) = I(Y; X)`, unlike KL.

**Bounds.**

- `0 \le I(X; Y) \le \min(H(X), H(Y))`.
- `I(X; Y) = 0` iff `X \perp\!\!\!\perp Y` (independent).
- `I(X; X) = H(X)` (knowing `X` perfectly tells you all of `H(X)`).

**ML angle.** Mutual information shows up in:

- **Feature selection.** Pick features with high `I(X_i; Y)` — they're informative about the label.
- **Information bottleneck.** Trade off `I(Z; Y)` (predictive) against `I(Z; X)` (compressed). A theoretical lens on representation learning.
- **InfoNCE / contrastive learning.** SimCLR, MoCo, CLIP train representations to maximize a lower bound on mutual information between augmentations or between modalities.
- **Variational autoencoders.** `\beta`-VAEs add a penalty proportional to `I(Z; X)` to learn disentangled representations.
- **Self-supervised pre-training.** Predict masked or future content — high `I(\text{visible}; \text{hidden})` between the two splits is what makes the task informative.


In [ ]:
# Mutual information from joint distribution
joint = np.array([
    [0.10, 0.20, 0.10],
    [0.05, 0.20, 0.10],
    [0.05, 0.10, 0.10],
])
print("Joint p(X, Y):"); print(joint); print(f"Total: {joint.sum():.4f}")

# Marginals
p_X = joint.sum(axis=1)
p_Y = joint.sum(axis=0)

print(f"\np_X = {p_X}")
print(f"p_Y = {p_Y}")

# Entropies
H_X  = entropy(p_X)
H_Y  = entropy(p_Y)
H_XY = entropy(joint.flatten())
I    = H_X + H_Y - H_XY

print(f"\nH(X)    = {H_X:.4f}")
print(f"H(Y)    = {H_Y:.4f}")
print(f"H(X, Y) = {H_XY:.4f}")
print(f"I(X; Y) = {I:.4f}  (= D_KL(joint || marginals * marginals))")

# Verify with the KL form
prod_marg = np.outer(p_X, p_Y)
I_kl = kl_divergence(joint.flatten(), prod_marg.flatten())
print(f"I(X; Y) via D_KL = {I_kl:.4f}  (should match)")


## Jensen's inequality — the workhorse

A convex function `f` lies *below* every chord. Equivalently, for a random variable `X`:

$$
f(\mathbb{E}[X]) \le \mathbb{E}[f(X)]
$$

For **concave** `f` the inequality flips:

$$
f(\mathbb{E}[X]) \ge \mathbb{E}[f(X)]
$$

This is **Jensen's inequality**, and it's the source of probably half the proofs in probability and information theory.

**Why it matters here.**

- **KL divergence is non-negative.** Apply Jensen to the concave function `-\log`:
  $$
  -D_{\text{KL}}(p \,\|\, q) = \mathbb{E}_{X \sim p}\!\left[\log \tfrac{q(X)}{p(X)}\right] \le \log \mathbb{E}_{X \sim p}\!\left[\tfrac{q(X)}{p(X)}\right] = \log 1 = 0
  $$
  So `D_{\text{KL}}(p \,\|\, q) \ge 0`. Three lines, profound consequences.
- **The Evidence Lower Bound (ELBO).** Variational inference uses Jensen to bound `\log p(x)` from below by a tractable expression, then maximizes that bound. The entire VAE training objective is a Jensen-derived inequality.
- **Concavity of entropy.** `H(p)` is a concave function of `p`. Hence Bayesian model averaging (a mixture) has entropy *higher* than the mixture-weighted average of the components' entropies.
- **Convexity of negative log-likelihood for exponential family.** Makes MLE for exponential-family distributions a convex optimization problem — solved efficiently, no bad local minima.

Jensen is one of those identities you'll apply over and over without explicitly invoking it. Once you spot the move (concave function, expectation moves in), the proofs that seemed magical become natural.


## Information theory bridge to ML losses

Bring it all together. The information-theoretic interpretation of the standard losses:

**Cross-entropy classification loss.** Minimizing `H(p_{\text{data}}, q_\theta)` over `\theta` is equivalent to minimizing `D_{\text{KL}}(p_{\text{data}} \,\|\, q_\theta)` (since `H(p_{\text{data}})` is constant). You're pulling the model's predicted distribution as close as possible to the empirical one.

**KL-regularized RL (TRPO, PPO).** Add `\beta \cdot D_{\text{KL}}(\pi_\theta \,\|\, \pi_{\text{old}})` to the policy gradient objective so updates can't move the policy too far per step. KL acts as a trust-region constraint.

**RLHF.** Reward = (reward model score) - `\beta \cdot D_{\text{KL}}(\pi_\theta \,\|\, \pi_{\text{SFT}})`. The KL penalty keeps the RLHF-tuned model from drifting too far from the supervised fine-tuned baseline.

**Distillation.** Train student `q_\theta` to match teacher `p_{\text{teacher}}` by minimizing `D_{\text{KL}}(p_{\text{teacher}} \,\|\, q_\theta)`. The teacher's soft probabilities carry more information than hard labels — "dark knowledge."

**Variational inference / ELBO / VAEs.** Maximize `\mathbb{E}_q[\log p(x | z)] - D_{\text{KL}}(q(z | x) \,\|\, p(z))`. The first term is reconstruction; the second is a KL regularizer that keeps the posterior close to the prior.

**InfoNCE (contrastive learning).** A lower bound on `I(z_i; z_j)` for positive pairs `(i, j)`. Maximizing it pushes positives together and negatives apart in representation space.

**Information bottleneck.** Train representations to maximize `I(Z; Y)` while minimizing `I(Z; X)` — extract only the information relevant to predicting `Y`. A theoretical lens, sometimes operationalized in practice.

These all share the same vocabulary: entropies, cross-entropies, KL divergences. Once you see them, you can read modern ML papers fluently.


## Where this appears in ML

Information theory is the second-most-used branch of math in modern ML, after linear algebra. Every probabilistic objective lives here.

- **Cross-entropy classification.** Standard loss for every classification model. Multi-class softmax + cross-entropy *is* `\text{NLL} = D_{\text{KL}}(p_{\text{empirical}} \,\|\, q_\theta)`.
- **Binary cross-entropy / log loss.** Same story, two-class version. Used in logistic regression, GANs (discriminator), binary classification.
- **Label smoothing.** Replace one-hot `(0, ..., 0, 1, 0, ..., 0)` with `(\epsilon, ..., 1 - (K-1)\epsilon, ..., \epsilon)`. Adds entropy to the target → softens the cross-entropy gradient → less overconfidence. Pure information-theory move.
- **VAE training.** Maximize the ELBO: reconstruction (cross-entropy or Gaussian NLL) minus `D_{\text{KL}}(q(z | x) \,\|\, p(z))`.
- **Diffusion model training.** Each denoising step has an ELBO-like KL objective; closed-form for Gaussian conditionals.
- **PPO / TRPO.** KL-divergence trust regions on policy updates.
- **RLHF.** KL penalty against the SFT reference model.
- **Distillation.** Student matches teacher's softmax distribution via KL (sometimes with temperature scaling).
- **Contrastive learning (SimCLR, MoCo, CLIP).** Maximize InfoNCE — a mutual information lower bound between views or modalities.
- **Information bottleneck networks.** Explicitly regularize `I(Z; X)` to learn minimal sufficient representations.
- **Mutual information for feature selection.** Pick features `X_i` with highest `I(X_i; Y)`.
- **Information gain in decision trees.** Splits chosen to maximize `H(Y) - H(Y | \text{split})` — exactly mutual information at the split.
- **Maximum-entropy modeling.** Choose `p` to maximize entropy subject to moment constraints. Equivalent to fitting an exponential-family distribution.
- **Bits-per-character / bits-per-token in language models.** A direct reading of cross-entropy in compression units. SOTA reporting in language modeling uses bits-per-character.
- **Calibration.** Compare predicted distribution to empirical via reliability diagrams and expected calibration error — entropy-aware variants of these include "Brier score decomposition."

Next notebook: **sampling, the Central Limit Theorem, and hypothesis testing** — the law of large numbers, the CLT, Monte Carlo, and A/B testing. Closing the probability repo and bridging to applied statistics.
